# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we examine all available record sets and their fields using their `@id` values, which uniquely identify dataset entities.

In [ ]:
# List all record sets and their @id values
record_sets = []

# The Croissant schema exposes record sets via metadata.recordSet, but some schemas use a property called 'recordSet'.
for obj in dataset.metadata.to_json().get('recordSet', []):
    record_sets.append(obj['@id'])

if len(record_sets) == 0:
    # If record sets are not accessible in metadata, fallback to scanning the dataset object
    print("Warning: No record sets found in metadata.")
    # For demonstration, we'll try to infer at least one record set from available mlcroissant access:
    record_sets = dataset.list_record_sets()
    print("Detected record sets via mlcroissant API:", record_sets)
else:
    print("Record sets @ids found in metadata:", record_sets)

# Print available fields of each record set (fields are accessed by @id)
for record_set_id in record_sets:
    print(f"\nRecord set: {record_set_id}")
    # Get fields for the record set
    fields = dataset.list_fields(record_set=record_set_id)
    print(f"Fields in record set '{record_set_id}': {fields}")

    # Optionally show sample records
    sample_records = list(dataset.records(record_set=record_set_id))[:3]
    for i, rec in enumerate(sample_records):
        print(f"Sample record {i+1}: {rec}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract data from all detected record sets, referencing them by their `@id`, and load them into pandas DataFrames for further processing.

In [ ]:
dataframes = {}

# Load records for each record set
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set '{record_set_id}' loaded with shape: {df.shape}")
    else:
        print(f"No records found for record set '{record_set_id}'.")

# Show columns and head for the first available DataFrame
if dataframes:
    sample_record_set_id = next(iter(dataframes))
    print(f"\nColumns for '{sample_record_set_id}':")
    print(dataframes[sample_record_set_id].columns.tolist())
    print("\nFirst five rows:")
    display(dataframes[sample_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this section, we'll:
- Filter records by a numeric field (e.g., GAD-7 score).
- Normalize scores.
- Group data by an attribute (e.g., gender) for further analysis.

All attributes referenced are by their column `@id`s, as required.

In [ ]:
# Choose a record set and field @id based on the overview above

# Example record set and field IDs (these will be the actual @ids from the dataset):
chosen_record_set_id = sample_record_set_id
df = dataframes[chosen_record_set_id]

# Try to find numeric fields available in this record set
numeric_field_id = None
for col in df.columns:
    if "gad7_score" in col.lower() or "phq9_score" in col.lower() or "psq_score" in col.lower() or "score" in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is not None:
    # Filtering, normalizing, grouping
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a demographic attribute (e.g., gender)
    group_field_id = None
    for col in df.columns:
        if "gender" in col.lower():
            group_field_id = col
            break
    
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print("No grouping field (e.g., gender) found for this record set.")
else:
    print("No numeric field suitable for EDA found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and, if available, compare means across demographic groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using `mlcroissant`.
- Inspected record sets and referenced all entities by their `@id`, in line with Croissant schema standards.
- Extracted and processed survey scores and demographic information; visualized data distributions and key relationships.
- Dataset provides valuable insight into mental health trends, with robust structure and FAIR compliance, demonstrating best practices for AI-ready data infrastructure in Africa.